## Seção 4.3 - Análise da sazonalidade da qualidade do ar no Brasil

Este notebook reproduz a figura 35 da seção 4.3 do Relatório Anual de Qualidade do Ar

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


### Figura 35 - Mapa interativo da sazonalidade de CO, NO₂, SO₂, MP₂,₅, MP₁₀ e O₃ no Brasil.

Mapa interativa que permite selecionar um poluente e uma métrica utilizada para descrever diferentes comportamentos sazonais nos dados:
- A força relativa da sazonalidade: Quanto da variabilidade total das concentrações de um determinado poluente pode ser explicada pelo padrão sazonal.
- Índice de Markham (MSI): Quanto a sazonalidade é concentrada em poucos meses.
- Amplitude: mostra a diferença entre o mês mais poluído e o menos poluído.

A legenda apresenta um grandiente contínuo no qual azul representam os valores mais baixos e vermelho os mais altos referentes a métrica escolhida

Ao clicar no ponto da estação é possível verificar o ID Estação, nome da estação, Período de monitoramento, anos inválidos, amplitude, força relativa, índice de markham, mês de maior média mensal e mês de menor média mensal. Estações sem período suficiente para a análise são apresentadas em cinza.

> **Pré-requisito:** este mapa depende dos arquivos `{poluente}_stations.geojson` gerados por `scripts/seasonality_analisys.ipynb`. Execute esse notebook primeiro (ele lê os dados diretamente da URL hospedada e salva os GeoJSONs em `_static/preprocessed_seasonality/`) antes de rodar a célula abaixo.

In [1]:
import re
import json
from pathlib import Path
from IPython.display import HTML

'''  >>> CONFIGURAÇÃO OBRIGATÓRIA <<<
Ajuste SEASONALITY_DIR para a pasta onde os arquivos {poluente}_stations.geojson foram
gerados no seu computador por scripts/seasonality_analisys.ipynb antes de executar este
script. Como o código é compartilhado via Git, este caminho varia entre usuários. '''
SEASONALITY_DIR = Path("../_static/preprocessed_seasonality/")

POLUENTES = ["O3", "CO", "NO2", "MP25", "MP10", "SO2"]

def to_safe_key(pol):
    return re.sub(r'[^a-z0-9]+', '', pol.lower())

# Carrega, em Python, todos os GeoJSONs de sazonalidade disponíveis localmente e
# embute os dados no HTML — evita depender de fetch() em tempo real, que não
# funciona quando o HTML é aberto como arquivo local (file://) fora de um servidor.
data = {}
for pol in POLUENTES:
    fpath = SEASONALITY_DIR / f"{to_safe_key(pol)}_stations.geojson"
    if fpath.exists():
        with open(fpath, encoding="utf-8") as f:
            data[pol] = json.load(f)

data_json = json.dumps(data, ensure_ascii=False)

html_template = r"""
<style>
#map-controls-season {
  display:flex;
  flex-wrap:nowrap;
  gap:12px;
  align-items:center;
  margin-bottom:12px;
  font-family:Arial, sans-serif;
}

.control {
  display:flex;
  align-items:center;
  gap:8px;
  height:40px;
  white-space:nowrap;
}

.control-label { font-size:14px; font-weight:500; color:#333; }

.select-wrap { position:relative; display:inline-block; width:160px; height:32px; }
.select-wrap select {
  appearance:none;
  display:block;
  width:100%;
  height:100%;
  padding:6px 34px 6px 10px;
  font-size:14px;
  border-radius:6px;
  border:1px solid #999;
  background:#f9f9f9;
  cursor:pointer;
}
.select-wrap::after{
  content:"";
  position:absolute;
  pointer-events:none;
  top:50%;
  transform:translateY(-50%);
  right:10px;
  width:12px;
  height:12px;
  background-image: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 10 6'><path fill='%23333' d='M0 0l5 6 5-6z'/></svg>");
  background-repeat:no-repeat;
  background-size:12px 12px;
}

.radio-group { display:flex; gap:12px; align-items:center; }

/* botão à direita */
.control-action {
  margin-left:auto;
  display:flex;
  align-items:center;
  gap:8px;
}
.control-button {
  height:32px;
  padding:0 16px;
  line-height:32px;
  border-radius:6px;
  border:1px solid #005a9e;
  background:#0078d7;
  color:#fff;
  cursor:pointer;
  font-weight:600;
  text-align:center;
}
.control-button:hover{ background:#005a9e; }

/* status simples */
#status-season { font-size:13px; color:crimson; margin-left:8px; }

/* mapa */
#leafletSeasonMap{ width:100%; height:640px; border:1px solid #ddd; display:none; margin-top:8px; }

/* legenda customizada (gradiente) */
.leaflet-control.custom-legend { background:white; padding:8px; border-radius:4px; box-shadow:0 1px 4px rgba(0,0,0,0.2); font-size:13px; }
.legend-gradient { height:12px; width:200px; border-radius:3px; margin:6px 0; display:block; }
.legend-labels { display:flex; justify-content:space-between; gap:8px; font-size:12px; }
</style>

<div id="map-controls-season">
  <div class="control">
    <span class="control-label">Poluente:</span>
    <div class="select-wrap">
      <select id="sel-poll-season">
        <option>O3</option>
        <option>CO</option>
        <option>NO2</option>
        <option>MP25</option>
        <option>MP10</option>
        <option>SO2</option>
      </select>
    </div>
  </div>

  <div class="control">
    <span class="control-label">Visualizar:</span>
    <div class="radio-group" title="Escolha a métrica a ser mostrada no mapa">
      <label style="display:inline-flex;align-items:center;gap:6px;">
        <input type="radio" name="seasonMetric" value="relative_strength" checked> Força relativa
      </label>
      <label style="display:inline-flex;align-items:center;gap:6px;">
        <input type="radio" name="seasonMetric" value="msi"> MSI
      </label>
      <label style="display:inline-flex;align-items:center;gap:6px;">
        <input type="radio" name="seasonMetric" value="amplitude"> Amplitude
      </label>
    </div>
  </div>

  <div class="control-action">
    <button id="btn-load-season" class="control-button">Gerar mapa</button>
    <span id="status-season"></span>
  </div>
</div>

<div id="leafletSeasonMap"></div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
(function(){
  const DATA = __DATA__;
  const selP = document.getElementById('sel-poll-season');
  const btn  = document.getElementById('btn-load-season');
  const status = document.getElementById('status-season');
  const mapDiv = document.getElementById('leafletSeasonMap');

  const tilesDefs = {
    "OpenStreetMap": { url: 'https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', attribution:'© OpenStreetMap' },
    "CartoDB Positron": { url: 'https://cartodb-basemaps-a.global.ssl.fastly.net/light_all/{z}/{x}/{y}.png', attribution:'© OpenStreetMap, © CartoDB' },
    "Esri WorldImagery": { url: 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attribution:'Tiles © Esri' }
  };

  let map = null;
  let relLayer = null;
  let msiLayer = null;
  let ampLayer = null;
  let baseLayers = {};
  let layerControl = null;
  let legendControl = null;

  function ensureMap(){
    if(map) return;
    map = L.map('leafletSeasonMap').setView([-15.78, -47.9], 4);
    Object.keys(tilesDefs).forEach((name, idx) => {
      const def = tilesDefs[name];
      baseLayers[name] = L.tileLayer(def.url, { maxZoom: 19, attribution: def.attribution });
      if(idx===0) baseLayers[name].addTo(map);
    });
    layerControl = L.control.layers(baseLayers, {}, { collapsed: false }).addTo(map);
  }

  const monthMapENtoPT = {
    "January":"janeiro","February":"fevereiro","March":"março","April":"abril","May":"maio","June":"junho",
    "July":"julho","August":"agosto","September":"setembro","October":"outubro","November":"novembro","December":"dezembro"
  };

  function fmt3(val){
    if(val===null || val===undefined || val === "" || isNaN(Number(val))) return "n/a";
    return Number(val).toFixed(3);
  }

  const jetStops = [
    {t:0.00, c:"#00007F"},
    {t:0.15, c:"#0000FF"},
    {t:0.35, c:"#00FFFF"},
    {t:0.60, c:"#FFFF00"},
    {t:0.85, c:"#FF7F00"},
    {t:1.00, c:"#7F0000"}
  ];

  function hexToRgb(hex){ hex = hex.replace('#',''); if(hex.length===3) hex = hex.split('').map(x=>x+x).join(''); const n = parseInt(hex,16); return [(n>>16)&255, (n>>8)&255, n&255]; }
  function rgbToHex(r,g,b){ return '#'+[r,g,b].map(v=>{ const s=v.toString(16); return s.length===1 ? '0'+s : s; }).join(''); }

  function interpJet(t){
    if(t<=0) return jetStops[0].c;
    if(t>=1) return jetStops[jetStops.length-1].c;
    for(let i=0;i<jetStops.length-1;i++){
      const a = jetStops[i], b = jetStops[i+1];
      if(t >= a.t && t <= b.t){
        const localT = (t - a.t) / (b.t - a.t || 1);
        const ra = hexToRgb(a.c), rb = hexToRgb(b.c);
        const r = Math.round(ra[0] + (rb[0]-ra[0])*localT);
        const g = Math.round(ra[1] + (rb[1]-ra[1])*localT);
        const bl= Math.round(ra[2] + (rb[2]-ra[2])*localT);
        return rgbToHex(r,g,bl);
      }
    }
    return jetStops[jetStops.length-1].c;
  }

  function colorForMetric(val, minV, maxV){
    if(val === null || val === undefined || isNaN(Number(val))) return "#666666";
    const t = (Number(val) - minV) / ((maxV - minV) || 1);
    return interpJet(Math.max(0, Math.min(1, t)));
  }

  function buildPopupSeason(props){
    if(!props) return "";
    const ID_OEMA = props.ID_OEMA || props.id_oema || "n/a";
    const station = props.station || props.estacao || "n/a";
    const startY = ('start_year' in props) ? props.start_year : null;
    const endY   = ('end_year' in props) ? props.end_year : null;
    const periodo = (startY !== null && endY !== null) ? `${startY} - ${endY}` : "n/a";
    const n_valid = ('n_valid_years' in props) ? parseInt(props.n_valid_years || 0) : 0;
    const invalid_years = ('invalid_years' in props) ? parseInt(props.invalid_years || 0) : 0;
    const p_value = ('p_value' in props) ? fmt3(props.p_value) : "n/a";
    const amplitude = ('amplitude' in props) ? fmt3(props.amplitude) : "n/a";
    const relative_strength = ('relative_strength' in props) ? fmt3(props.relative_strength) : "n/a";
    const msi = ('msi' in props) ? fmt3(props.msi) : "n/a";
    let max_month = props.max_month || props.maxMonth || "";
    let min_month = props.min_month || props.minMonth || "";
    if(monthMapENtoPT[max_month]) max_month = monthMapENtoPT[max_month];
    if(monthMapENtoPT[min_month]) min_month = monthMapENtoPT[min_month];

    return `<b>ID Estação:</b> ${station}<br/>
            <b>Estação:</b> ${ID_OEMA}<br/>
            <b>Período:</b> ${periodo}<br/>
            <b>Anos inválidos:</b> ${invalid_years}<br/>
            <b>Amplitude:</b> ${amplitude}<br/>
            <b>Força relativa:</b> ${relative_strength}<br/>
            <b>Índice de Markham (MSI):</b> ${msi}<br/>
            <b>Mês max:</b> ${max_month}<br/>
            <b>Mês min:</b> ${min_month}`;
  }

  function addLegendForMetric(minV, maxV, title){
    if(legendControl){ try{ legendControl.remove(); } catch(e){} legendControl = null; }
    legendControl = L.control({ position: 'bottomright' });
    legendControl.onAdd = function(){
      const div = L.DomUtil.create('div', 'leaflet-control custom-legend');
      div.innerHTML = `<strong>${title}</strong>`;
      const stopsHtml = [];
      const sampleCount = 7;
      for(let i=0;i<sampleCount;i++){
        const t = i/(sampleCount-1);
        const c = interpJet(t);
        stopsHtml.push(`<span style="display:inline-block;width:${100/sampleCount}%;height:12px;background:${c};"></span>`);
      }
      div.innerHTML += `<div class="legend-gradient" style="display:flex;overflow:hidden;border:1px solid #ddd;border-radius:3px;padding:0;">${stopsHtml.join('')}</div>`;

      let minLabel, maxLabel;
      if(title.includes("Força relativa")){
        minLabel = "0";
        maxLabel = "1";
      } else if(title.includes("MSI") || title.includes("Amplitude")){
        minLabel = "0";
        maxLabel = (isNaN(maxV) ? "n/a" : Number(maxV).toFixed(3));
      } else {
        minLabel = (isNaN(minV) ? "n/a" : Number(minV).toFixed(3));
        maxLabel = (isNaN(maxV) ? "n/a" : Number(maxV).toFixed(3));
      }

      div.innerHTML += `<div class="legend-labels"><span>${minLabel}</span><span>${maxLabel}</span></div>`;

      div.innerHTML += `<div style="margin-top:6px;display:flex;align-items:center;gap:6px;">
        <span style="display:inline-block;width:14px;height:14px;background:#cccccc;border:1px solid #222;border-radius:50%;"></span>
        <span>Período insuficiente</span>
      </div>`;

      return div;
    };
    legendControl.addTo(map);
  }

  btn.addEventListener('click', function(){
    try{
      status.textContent = '';
      mapDiv.style.display = 'block';
      ensureMap();

      if(relLayer){ try{ map.removeLayer(relLayer); } catch(e){} relLayer = null; }
      if(msiLayer){ try{ map.removeLayer(msiLayer); } catch(e){} msiLayer = null; }
      if(ampLayer){ try{ map.removeLayer(ampLayer); } catch(e){} ampLayer = null; }
      if(layerControl){ try{ layerControl.remove(); } catch(e){} layerControl = L.control.layers(baseLayers, {}, { collapsed: false }).addTo(map); }

      const polSel = selP.value;
      const gj = DATA[polSel];
      if(!gj || !gj.features || gj.features.length === 0){
        status.textContent = `Dados não encontrados para ${polSel}`;
        return;
      }

      const validFeatures = gj.features.filter(f => {
        const n_valid = parseInt(f.properties?.n_valid_years || 0);
        return n_valid > 0;
      });

      const relVals = validFeatures.map(f => Number(f.properties.relative_strength)).filter(v => !isNaN(v));
      const msiVals = validFeatures.map(f => Number(f.properties.msi)).filter(v => !isNaN(v));
      const ampVals = validFeatures.map(f => Number(f.properties.amplitude)).filter(v => !isNaN(v));

      if(relVals.length === 0 && msiVals.length === 0 && ampVals.length === 0){
        status.textContent = `Nenhum valor válido encontrado para ${polSel}`;
        return;
      }

      const relMin = 0;
      const relMax = 1;
      const msiMin = 0;
      const msiMax = msiVals.length ? Math.max(...msiVals) : NaN;
      const ampMin = 0;
      const ampMax = ampVals.length ? Math.max(...ampVals) : NaN;

      relLayer = L.geoJSON(gj, {
        pointToLayer: (f, latlng) => {
          const p = f.properties || {};
          const n_valid = parseInt(p.n_valid_years || 0);
          const color = (n_valid === 0) ? "#cccccc" : colorForMetric(Number(p.relative_strength), relMin, relMax);
          return L.circleMarker(latlng, { radius:7, fillColor: color, color:"#222", weight:0.6, fillOpacity:0.95 });
        },
        onEachFeature: (f, layer) => { layer.bindPopup(buildPopupSeason(f.properties)); }
      });

      msiLayer = L.geoJSON(gj, {
        pointToLayer: (f, latlng) => {
          const p = f.properties || {};
          const n_valid = parseInt(p.n_valid_years || 0);
          const color = (n_valid === 0) ? "#cccccc" : colorForMetric(Number(p.msi), msiMin, msiMax);
          return L.circleMarker(latlng, { radius:7, fillColor: color, color:"#222", weight:0.6, fillOpacity:0.95 });
        },
        onEachFeature: (f, layer) => { layer.bindPopup(buildPopupSeason(f.properties)); }
      });

      ampLayer = L.geoJSON(gj, {
        pointToLayer: (f, latlng) => {
          const p = f.properties || {};
          const n_valid = parseInt(p.n_valid_years || 0);
          const color = (n_valid === 0) ? "#cccccc" : colorForMetric(Number(p.amplitude), ampMin, ampMax);
          return L.circleMarker(latlng, { radius:7, fillColor: color, color:"#222", weight:0.6, fillOpacity:0.95 });
        },
        onEachFeature: (f, layer) => { layer.bindPopup(buildPopupSeason(f.properties)); }
      });

      const metric = document.querySelector('input[name="seasonMetric"]:checked').value;
      if(metric === "relative_strength"){
        relLayer.addTo(map);
        layerControl.addOverlay(relLayer, "Força relativa (visível)");
        layerControl.addOverlay(msiLayer, "MSI (oculta)");
        layerControl.addOverlay(ampLayer, "Amplitude (oculta)");
        addLegendForMetric(relMin, relMax, "Força relativa");
      } else if(metric === "msi"){
        msiLayer.addTo(map);
        layerControl.addOverlay(msiLayer, "MSI (visível)");
        layerControl.addOverlay(relLayer, "Força relativa (oculta)");
        layerControl.addOverlay(ampLayer, "Amplitude (oculta)");
        addLegendForMetric(msiMin, msiMax, "Índice de Markham (MSI)");
      } else {
        ampLayer.addTo(map);
        layerControl.addOverlay(ampLayer, "Amplitude (visível)");
        layerControl.addOverlay(relLayer, "Força relativa (oculta)");
        layerControl.addOverlay(msiLayer, "MSI (oculta)");
        addLegendForMetric(ampMin, ampMax, "Amplitude");
      }

      const visibleLayer = (metric==="relative_strength") ? relLayer : ((metric==="msi") ? msiLayer : ampLayer);
      const fg = L.featureGroup([visibleLayer].filter(Boolean));
      if(fg.getBounds && fg.getBounds().isValid()){
        map.fitBounds(fg.getBounds(), { padding:[20,20] });
      }

      status.textContent = '';
    } catch(err){
      console.error(err);
      status.textContent = 'Erro ao gerar o mapa sazonal (ver console)';
    }
  });
})();
</script>
"""

html_code = html_template.replace("__DATA__", data_json)

HTML(html_code)

In [ ]:
# Para salvar a figura e abrir no navegador, ajuste o caminho de saída conforme necessário.

import os
output_dir = "outputs"
output_path = os.path.join(output_dir, "figura35.html")

# 1. Create the 'outputs' directory automatically if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# 2. Write the file
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_code)

# 3. Open in browser
webbrowser.open(output_path)


True